<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Appendix F: *Raster Processing*
##### Version Number: 4.0
---
### Contents  
> 1. ** 
> 2. **
---
### Notes

- Very early planning stages for hot spot analysis of daily california NDVI raster data
---
### Inputs
- 
---
### Outputs  
- 
---
### User Created Dependencies  
---

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
import pandas as pd
import rasterio
import datetime
from datetime import datetime, timedelta
from rasterio.windows import from_bounds
import numpy as np
import glob

import warnings

warnings.filterwarnings("ignore")

## Raster Processing Notes

pygeoprocessing.raster_calculator()

pygeoprocessing.raster_calculator(
    base_raster_list,
    local_op,        # function to apply on pixel values
    target_raster_path,
    target_datatype,
    nodata_list=None,
    dtype_mapping=None
)

## Raster Processing Algorithm
0) **Split**
    - Split rasters into each sampling grid area by lat and lon 

1) **Global reference**
    - Compute the global statistic (mean, median, or max and std) to provide context for identifying hotspots/coldspots.

2) **Split raster into overlapping pieces**
    - Overlapping ensures that hotspots near piece edges aren’t missed.
    - n is determined by processor/memory limits.

3) **Assign piece value**
    - Use mean, median, or max and std to summarize each piece.
    - This reduces data resolution for hotspot detection while preserving extremes.

4) **Compare piece values**
    - Use both neighboring pieces and global mean to detect relative extremes.

5) **Label hotspots/coldspots**
    - Each piece is marked as hot, cold, or neutral.

6) **Summarize clusters**
    - Count total hotspots/coldspots.
    - Optionally compute largest cluster size or number of significant clusters for further analysis.

## Hotspot Algorithm

1) **global hot spot**
    - Formula - cell_value > global_mean + k * global_std
2) **local hot spot** 
    - Formula - cell_value > local_mean + k × local_std
3) **Variables**
    - `cell_value` = The value of the raster at the center cell you are currently evaluating.
    - `local_mean` = The average value of all cells in the neighborhood window around the center cell
    - `global_mean` = The average value of all cells in the raster
    - `global_std` = The standard deviation of all the values
    - `local_std` = The standard deviation of the neighborhood values
    - `k` = A multiplier that defines how extreme a value must be to count as a hotspot.
        - Example: k = 2 → more strict: 2 standard deviations above.

- VIIRS-Land_v001_NPP13C1_S-NPP_20180101_c20240125132403_CA
- VIIRS-Land_v001_NPP13C1_S-NPP_20190101_c20240125132355_CA
- VIIRS-Land_v001_JP113C1_NOAA-20_20230101_c20240123205954_CA

In [3]:
data = pd.read_csv('../data/processed/samples_projected.csv')

In [4]:
sub = data[['Date','grid_id','minimum_x','maximum_x','maximum_y','minimum_y']]

In [5]:
sub['NDVI_mean_difference'] = np.nan
sub['NDVI_local_max'] = np.nan
sub['NDVI_local_min'] = np.nan
sub['NDVI_local_std'] = np.nan
sub['NDVI_global_min'] = np.nan
sub['NDVI_global_max'] = np.nan
sub['NDVI_global_std'] = np.nan
sub['NDVI_global_mean'] = np.nan

In [6]:
years = list(range(2025,2026))

dates = pd.date_range(
    start="2018-01-01",
    end="2025-01-23",
    freq="D"
)

days = dates.strftime("%Y%m%d").tolist()

In [7]:
years

[2025]

In [8]:
sub['Date'] = pd.to_datetime(sub['Date'])

In [9]:
sub['Year'] = sub['Date'].dt.year

In [10]:
sub = sub[sub['Year'] == years[0]]

In [11]:
sub

,Date,grid_id,minimum_x,maximum_x,maximum_y,minimum_y,NDVI_mean_difference,NDVI_local_max,NDVI_local_min,NDVI_local_std,NDVI_global_min,NDVI_global_max,NDVI_global_std,NDVI_global_mean,Year
603452,2025-01-01,150,-121.186206,-120.658436,37.948635,37.530840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
603453,2025-01-01,70,-115.116615,-114.587513,34.948324,34.512049,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
603454,2025-01-01,71,-114.616109,-114.084664,34.925781,34.487433,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
603455,2025-01-01,169,-122.788851,-122.246631,38.757062,38.332402,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
603456,2025-01-01,229,-120.144289,-119.595853,41.262788,40.847753,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
608875,2025-01-23,158,-122.773355,-122.234214,38.343356,37.918781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
608876,2025-01-23,157,-123.299907,-122.758030,38.332402,37.905603,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
608877,2025-01-23,166,-118.565476,-118.030579,38.355642,37.934488,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
608878,2025-01-23,168,-123.318341,-122.773355,38.746045,38.319150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025


In [12]:
for year in years:
    
    raster_folder = '../data/raw/ndvi_california/ndvi_clipped_' + f'{year}'
    
    day = ''
    
    for index,row in sub.iterrows():
        
        date = row['Date']
        
        if date.strftime("%Y%m%d") != day:
            day = date.strftime("%Y%m%d")
        
            raster_path = '/VIIRS-Land_v001_JP113C1_NOAA-20_' + f'{day}'
    
            prefix = raster_folder + raster_path
            pattern = prefix + "*.tif"   # or "*" if extension varies
            
            matches = glob.glob(pattern)           
            
            if not matches:
                print(f"No file found for {day}, skipping...")
                continue
    
            raw_file = matches[0]
        
            if date.day == 1:
                print('processing ' + raw_file + '...')
    
            with rasterio.open(matches[0]) as src:
                raster = src.read
                nodata = src.nodata
    
                if nodata is not None:
                    raster = np.where(raster == nodata, np.nan, raster)
    
            with rasterio.open(matches[0]) as src:
                raster = src.read(1)
                nodata = src.nodata

                if nodata is not None:
                    raster = np.where(raster == nodata, np.nan, raster)
                    
            if raster.size == 0 or np.all(np.isnan(raster)):
                global_mean = np.nan  # or None
                global_min = np.nan
                global_max = np.nan
                global_std = np.nan
            else:
                global_mean_val = np.nanmean(raster)
                global_min_val = np.nanmin(raster)
                global_max_val = np.nanmax(raster)
                global_std_value = np.nanstd(raster)
            
        if not matches:
            print(f"No file found for {day}, skipping...")
            continue
            
        #print('Date = ' + day + 'Grid = ', + row['grid_id'])
        xmin = row['minimum_x']
        xmax = row['maximum_x']
        ymin = row['minimum_y']
        ymax = row['maximum_y']
        
        with rasterio.open(matches[0]) as src:
            window = from_bounds(xmin, ymin, xmax, ymax, src.transform)
            subset = src.read( 1,
                window=((int(window.row_off), int(window.row_off + math.ceil(window.height))),
                (int(window.col_off), int(window.col_off + math.ceil(window.width))))
            )
        
            nodata = src.nodata  # get from rasterio

            if nodata is not None:
                subset = np.where(subset == nodata, np.nan, subset)

            mean_val = np.nanmean(subset)
            
            if subset.size == 0 or np.all(np.isnan(subset)):
                local_max = np.nan  # or None
                local_min = np.nan
                local_mean = np.nan
                local_std = np.nan
            else:
                local_max = np.nanmax(subset)
                local_min = np.nanmin(subset)
                local_mean = np.nanmean(subset)
                local_std = np.nanstd(subset)
                
            sub.loc[(sub.index == index) & (sub['Date'] == date), 'NDVI_local_mean'] = local_mean
            sub.loc[(sub.index == index) & (sub['Date'] == date), 'NDVI_local_max'] = local_max
            sub.loc[(sub.index == index) & (sub['Date'] == date), 'NDVI_local_min'] = local_min
            sub.loc[(sub.index == index) & (sub['Date'] == date), 'NDVI_local_std'] = local_std 
            sub.loc[(sub.index == index) & (sub['Date'] == date), 'NDVI_global_min'] = global_min_val
            sub.loc[(sub.index == index) & (sub['Date'] == date), 'NDVI_global_max'] = global_max_val
            sub.loc[(sub.index == index) & (sub['Date'] == date), 'NDVI_global_std'] = global_std_value
            sub.loc[(sub.index == index) & (sub['Date'] == date), 'NDVI_global_mean'] = global_mean_val
        

processing ../data/raw/ndvi_california/ndvi_clipped_2025\VIIRS-Land_v001_JP113C1_NOAA-20_20250101_c20250103153010_CA.tif...


In [13]:
sub.to_csv('../data/raw/NDVI_mean_2025.csv',index=False)